In [0]:
# Cell: Check what folders exist
storage_account = "jeitstorageaccountdev"
container = "raw"
base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

print("=== Checking Taxi_Data folder ===")
try:
    display(dbutils.fs.ls(f"{base}/Taxi_Data/"))
except Exception as e:
    print(f"❌ Taxi_Data folder not found: {e}")
    print("\nTrying root of raw container:")
    display(dbutils.fs.ls(base))

=== Checking Taxi_Data folder ===


path,name,size,modificationTime
abfss://raw@jeitstorageaccountdev.dfs.core.windows.net/Taxi_Data/green/,green/,0,1771351566000
abfss://raw@jeitstorageaccountdev.dfs.core.windows.net/Taxi_Data/yellow/,yellow/,0,1771351501000


In [0]:
# Cell: Green Taxi Bronze with file-by-file normalization
from pyspark.sql.functions import col, lit, year, month
from functools import reduce

years = [2023, 2024, 2025]
base = "abfss://raw@jeitstorageaccountdev.dfs.core.windows.net/Taxi_Data"

# Get all green parquet files
green_files = []
for yr in years:
    try:
        files = dbutils.fs.ls(f"{base}/green/{yr}/")
        green_files.extend([f.path for f in files if f.name.endswith('.parquet')])
    except:
        print(f"⚠️  No green files for {yr}")

print(f"Found {len(green_files)} green parquet files")

# Type mapping for green taxi
green_type_mapping = {
    "VendorID": "long",
    "RatecodeID": "double",
    "PULocationID": "long",
    "DOLocationID": "long",
    "passenger_count": "double",
    "trip_distance": "double",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "ehail_fee": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "payment_type": "long",
    "trip_type": "long",
    "congestion_surcharge": "double"
}

# Read and normalize each file
def normalize_green_file(file_path):
    df = spark.read.format("parquet").load(file_path)
    for col_name, target_type in green_type_mapping.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, col(col_name).cast(target_type))
    return df

# Read all files with normalization
print("Reading and normalizing green files...")
green_dfs = [normalize_green_file(f) for f in green_files]

# Union all
green_bronze = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), 
    green_dfs
)

# Add partitioning columns
green_bronze = green_bronze \
    .withColumn("year", year("lpep_pickup_datetime")) \
    .withColumn("month", month("lpep_pickup_datetime"))

print(f"✅ Total green records: {green_bronze.count():,}")

# Save
green_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .saveAsTable("jeit_kg_dev.default.green_taxi_bronze")

print("✅ green_taxi_bronze saved!")

Found 35 green parquet files
Reading and normalizing green files...
✅ Total green records: 1,990,417
✅ green_taxi_bronze saved!


In [0]:
from pyspark.sql.functions import col, year, month

years = [2023, 2024, 2025]
base = "abfss://raw@jeitstorageaccountdev.dfs.core.windows.net/Taxi_Data"
table_name = "jeit_kg_dev.default.yellow_taxi_bronze"

yellow_type_mapping = {
    "VendorID": "double",
    "RatecodeID": "double",
    "PULocationID": "double",
    "DOLocationID": "double",
    "passenger_count": "double",
    "trip_distance": "double",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "payment_type": "double",
    "congestion_surcharge": "double",
    "airport_fee": "double"
}

spark.sql(f"DROP TABLE IF EXISTS {table_name}")

spark.conf.set("spark.sql.shuffle.partitions", "50")

for yr in years:
    print(f"Processing year {yr}...")

    try:
        files = dbutils.fs.ls(f"{base}/yellow/{yr}/")
        parquet_files = [f.path for f in files if f.name.endswith(".parquet")]
    except:
        print(f"No files for {yr}")
        continue

    for file_path in parquet_files:
        df = spark.read.parquet(file_path)

        # Cast immediately (this fixes INT64 vs DOUBLE mismatch)
        for c, t in yellow_type_mapping.items():
            if c in df.columns:
                df = df.withColumn(c, col(c).cast(t))

        df = df.withColumn("year", year("tpep_pickup_datetime")) \
               .withColumn("month", month("tpep_pickup_datetime"))

        (df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .partitionBy("year", "month")
            .saveAsTable(table_name)
        )

    print(f"✅ Finished {yr}")

print("✅ yellow_taxi_bronze completed safely!")


Processing year 2023...
✅ Finished 2023
Processing year 2024...
✅ Finished 2024
Processing year 2025...
✅ Finished 2025
✅ yellow_taxi_bronze completed safely!


In [0]:
green_count = spark.sql("SELECT COUNT(*) FROM jeit_kg_dev.default.green_taxi_bronze").collect()[0][0]
yellow_count = spark.sql("SELECT COUNT(*) FROM jeit_kg_dev.default.yellow_taxi_bronze").collect()[0][0]

print(f"\n✅ GREEN TAXI:  {green_count:,} records")
print(f"✅ YELLOW TAXI: {yellow_count:,} records")
print(f"\n   TOTAL:      {green_count + yellow_count:,} records")
print("\n" + "=" * 60)


✅ GREEN TAXI:  1,990,417 records
✅ YELLOW TAXI: 123,897,542 records

   TOTAL:      125,887,959 records



In [0]:
from pyspark.sql.functions import col
from functools import reduce

years = [2023, 2024, 2025]
colors = ["green", "yellow"]

# Get list of all parquet files for each year and color
parquet_files = []
for color in colors:
    for year in years:
        files = dbutils.fs.ls(f"abfss://raw@jeitstorageaccountdev.dfs.core.windows.net/Taxi_Data/{color}/{year}/")
        parquet_files.extend([f.path for f in files if f.name.endswith('.parquet')])

# Define target types for columns that vary across files
type_mapping = {
    "VendorID": "long",
    "RatecodeID": "double",
    "PULocationID": "long",
    "DOLocationID": "long",
    "passenger_count": "double",
    "ehail_fee": "double",
    "payment_type": "double",
    "trip_type": "double"
}

# Read and normalize each file
def normalize_schema(file_path):
    df = spark.read.format("parquet").load(file_path)
    for col_name, target_type in type_mapping.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, col(col_name).cast(target_type))
    return df

# Read all files and union them, allowing missing columns
dfs = [normalize_schema(f) for f in parquet_files]

df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dfs)

display(df.count())

125887959